# Dipole Fit Chi² Histograms — Gaia Category Comparison

This notebook displays histograms of the **chi² values from dipole fits** (`r:dipoleFluxChi2`
or equivalent column) for DIA sources flagged as dipoles (`r:isDipole == True`),
broken down by Gaia stellar category.

**Scientific motivation:**  
The chi² of the dipole fit model quantifies the quality of the PSF-subtraction residual
fit to a dipole template.  A distribution peaked near 1 indicates a good fit (genuine
dipole morphology); a broad or heavy tail signals either a poor fit, a non-dipolar
residual, or genuine astrophysical variability.

Comparing this distribution across Gaia categories reveals whether stable stars,
photometrically-undefined stable stars, and variable stars produce statistically
different dipole residuals — which would be a key diagnostic for the Fink alert
contamination hypothesis.

**Data source:** `*_src.parquet` files from `data_FINK_BLOCK_LC_01/`  
**Columns of interest:** `r:isDipole`, `r:dipoleFluxChi2` (or fallback candidates),
`r:band`, `r:psfFlux`, `r:scienceFlux`

**Categories:**
- `gaia_star_stable_hq` — high-quality Gaia stable stars (calibration targets)
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable, no photometric solution
- `gaia_star_variable` — Gaia variable stars (control group)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-19  
**Notebook:** 11d — inspired from 11_dipole_analysis.ipynb  
**Subject:** Rubin LSST — Fink alert broker — dipole chi² diagnostics

## 1. Imports & configuration

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input parquet files
DIR_FIGS = "figs_FINK_BLOCK_LC_11d"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Source categories to analyse ──────────────────────────────────────────────
CATEGORIES = {
    "gaia_star_stable_hq": {
        "label": "Gaia stable HQ",
        "short": "stable_hq",
        "color": "steelblue",
    },
    "gaia_nophotgstar_stable_unknown_parallax": {
        "label": "Gaia stable no-phot",
        "short": "stable_nophot",
        "color": "seagreen",
    },
    "gaia_star_variable": {
        "label": "Gaia variable",
        "short": "variable",
        "color": "firebrick",
    },
}

# ── Candidate chi2 column names (try in order) ────────────────────────────────
CHI2_CANDIDATES = [
    "r:dipoleFluxChi2",
    "r:dipoleChi2",
    "r:ip_diffim_DipoleFit_chi2",
    "r:ip_diffim_dipole_chi2",
]

# ── Histogram binning for chi2 ────────────────────────────────────────────────
# Chi2 values from a dipole fit with ~O(10) dof; most values near 1 for good fits
CHI2_BINS_FINE = np.linspace(0, 10, 51)  # fine binning for 0..10
CHI2_BINS_WIDE = np.linspace(0, 50, 51)  # wide binning for outlier exploration
CHI2_MAX_DISPLAY = 20.0  # clip x-axis for readability

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


def find_chi2_column(df: pd.DataFrame) -> str | None:
    """
    Return the name of the first chi2-like column present in *df*.
    Returns None if none of the candidates are found.
    """
    for col in CHI2_CANDIDATES:
        if col in df.columns:
            return col
    return None


def chi2_reduced(chi2_values, ndof):
    """Return reduced chi2 = chi2 / ndof (scalar or array)."""
    return np.asarray(chi2_values, dtype=float) / ndof


print("Utility functions defined.")

## 3. Load parquet data

We load the `*_src.parquet` files and keep only dipole-flagged sources
(`is_dipole == True`) for the chi² analysis.

In [ ]:
data_all = {}  # data_all[cat]  = full DataFrame (all sources)
data_dip = {}  # data_dip[cat]  = dipole-only DataFrame
chi2_col = {}  # chi2_col[cat]  = resolved chi2 column name

for cat in CATEGORIES:
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Minimal column guard ──────────────────────────────────────────────
    required = ["r:psfFlux", "r:band"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[WARN] {cat}: missing columns {missing} — skipped")
        continue

    # ── isDipole flag ─────────────────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: no r:isDipole column — all set to False")
        df["is_dipole"] = False

    # ── Resolve chi2 column ───────────────────────────────────────────────
    c2col = find_chi2_column(df)
    if c2col is None:
        print(f"[WARN] {cat}: no chi2 column found among {CHI2_CANDIDATES}")
        print(
            f"         available columns: {[c for c in df.columns if 'dipole' in c.lower() or 'chi' in c.lower()]}"
        )
    else:
        print(f"[INFO] {cat}: using chi2 column '{c2col}'")
        df["chi2"] = pd.to_numeric(df[c2col], errors="coerce")

    chi2_col[cat] = c2col
    data_all[cat] = df
    data_dip[cat] = df[df["is_dipole"]].copy()

    lbl = CATEGORIES[cat]["label"]
    n_tot = len(df)
    n_dip = len(data_dip[cat])
    n_chi2_ok = int(data_dip[cat]["chi2"].notna().sum()) if "chi2" in data_dip[cat].columns else 0
    print(
        f"  {lbl:40s}  "
        f"total={n_tot:6d}  dipoles={n_dip:5d} ({100 * n_dip / n_tot:.1f}%)  "
        f"chi2_valid={n_chi2_ok}"
    )

CATS_OK = list(data_all.keys())
print(f"\nCategories loaded: {CATS_OK}")

## 4. Column inspection — what chi² variants are present?

Print all dipole/chi2-related columns for the first loaded category to guide
manual column selection if the automatic search misses something.

In [ ]:
if CATS_OK:
    cat0 = CATS_OK[0]
    df0 = data_all[cat0]
    dip_cols = [
        c for c in df0.columns if any(kw in c.lower() for kw in ("dipole", "chi2", "chi", "fit", "model"))
    ]
    print(f"Dipole / chi2 related columns in '{cat0}' ({len(dip_cols)} found):")
    for c in sorted(dip_cols):
        n_valid = int(df0[c].notna().sum())
        dtype = df0[c].dtype
        print(f"  {c:<55s}  dtype={dtype}  valid={n_valid}")

In [ ]:
# ── Manual override (uncomment and adjust if needed) ──────────────────────────
# If the automatic column detection above failed, set the column name manually:
#
# MANUAL_CHI2_COL = "r:dipoleFluxChi2"   # <-- replace with the real column name
# for cat in CATS_OK:
#     if MANUAL_CHI2_COL in data_all[cat].columns:
#         data_all[cat]["chi2"] = pd.to_numeric(data_all[cat][MANUAL_CHI2_COL], errors="coerce")
#         data_dip[cat] = data_all[cat][data_all[cat]["is_dipole"]].copy()
#         chi2_col[cat] = MANUAL_CHI2_COL
#         print(f"  Manual override: {cat} → '{MANUAL_CHI2_COL}'")

print("Manual override block (commented out). Uncomment and edit if auto-detection failed.")

## 5. Basic chi² statistics per category

Summary table: mean, median, std, percentiles of the chi² distribution
for dipole-flagged sources, per Gaia category.

In [ ]:
rows_stats = []
for cat in CATS_OK:
    df_d = data_dip[cat]
    lbl = CATEGORIES[cat]["label"]

    if "chi2" not in df_d.columns:
        print(f"[SKIP stats] {cat}: no chi2 column")
        continue

    c2 = df_d["chi2"].dropna().values
    if len(c2) == 0:
        print(f"[SKIP stats] {cat}: all chi2 values are NaN")
        continue

    rows_stats.append(
        {
            "category": lbl,
            "N_dipoles": len(c2),
            "mean": round(np.mean(c2), 3),
            "median": round(np.median(c2), 3),
            "std": round(np.std(c2), 3),
            "p5": round(np.percentile(c2, 5), 3),
            "p25": round(np.percentile(c2, 25), 3),
            "p75": round(np.percentile(c2, 75), 3),
            "p95": round(np.percentile(c2, 95), 3),
            "p99": round(np.percentile(c2, 99), 3),
            "frac_chi2_gt10": round(float((c2 > 10).mean()), 4),
        }
    )

if rows_stats:
    df_stats = pd.DataFrame(rows_stats)
    print("Chi² statistics for dipole-flagged sources:")
    display(df_stats)
else:
    print("No chi² data available — check column names above.")

## 6. Chi² histograms — all categories overlaid

Three sub-figures:
1. **Raw histogram** (linear y-axis) — 0..20 range
2. **Normalised histogram** (density) — all categories on the same scale
3. **Log-scale** histogram — reveals tails beyond chi²=10

The theoretical chi² p.d.f. for `ndof` degrees of freedom is overplotted
as a dashed black curve to guide the eye.  The number of dof is not known
a priori from the parquet columns, so we estimate it from the median of the
distribution (`ndof_est ≈ median / 0.6745` for a chi² distribution,
since median ≈ ndof * (1 − 2/(9*ndof))^3 ≈ ndof for large ndof).

In [ ]:
# ── Collect chi2 arrays ───────────────────────────────────────────────────────
chi2_arrays = {}
for cat in CATS_OK:
    df_d = data_dip[cat]
    if "chi2" in df_d.columns:
        c2 = df_d["chi2"].dropna().values
        if len(c2) > 0:
            chi2_arrays[cat] = c2

if not chi2_arrays:
    print("No chi2 data — cannot plot. Check column detection above.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Dipole fit chi² distribution — all Gaia categories", fontsize=12, fontweight="bold", y=1.01)

    bins_fine = np.linspace(0, CHI2_MAX_DISPLAY, 80)

    # Estimate reference ndof from the first available category
    first_c2 = next(iter(chi2_arrays.values()))
    ndof_est = max(1, int(round(np.median(first_c2))))
    chi2_x = np.linspace(0.1, CHI2_MAX_DISPLAY, 300)
    chi2_pdf = stats.chi2.pdf(chi2_x, df=ndof_est)

    for ax_idx, (ax, (ylabel, yscale, normalise)) in enumerate(
        zip(
            axes,
            [
                ("N dipoles", "linear", False),
                ("Density (normalised)", "linear", True),
                ("N dipoles  [log]", "log", False),
            ],
        )
    ):
        for cat, c2 in chi2_arrays.items():
            lbl = CATEGORIES[cat]["label"]
            color = CATEGORIES[cat]["color"]
            ax.hist(
                c2,
                bins=bins_fine,
                density=normalise,
                alpha=0.55,
                color=color,
                label=f"{lbl} (N={len(c2):,})",
                histtype="stepfilled",
                edgecolor=color,
                linewidth=0.6,
            )

        # Theoretical chi2 PDF overlay (density panel only)
        if normalise:
            ax.plot(chi2_x, chi2_pdf, "k--", lw=1.5, label=f"$\\chi^2$(ndof={ndof_est}) theory", zorder=5)

        ax.axvline(1.0, color="grey", lw=0.8, ls=":", alpha=0.7, label="chi2=1")
        ax.set_xlabel("Dipole fit chi² ", fontsize=10)
        ax.set_ylabel(ylabel)
        ax.set_xlim(0, CHI2_MAX_DISPLAY)
        ax.set_yscale(yscale)
        if yscale == "log":
            ax.set_ylim(bottom=0.5)
        if ax_idx == 0:
            ax.legend(fontsize=8)
        elif ax_idx == 1:
            ax.legend(fontsize=8)

    plt.tight_layout()
    savefig("11d_chi2_all_categories")
    plt.show()

## 7. Chi² histograms — one panel per category

Individual panels for each Gaia category with finer control on axis limits.  
Each panel shows:
- histogram of chi² (filled, colour-coded by category)
- median and 95th percentile vertical lines
- theoretical chi²(ndof) curve (density mode)

In [ ]:
if chi2_arrays:
    n_cats = len(chi2_arrays)
    fig, axes = plt.subplots(1, n_cats, figsize=(5.5 * n_cats, 5), sharey=False, squeeze=False)
    fig.suptitle("Dipole fit chi² — per Gaia category", fontsize=12, fontweight="bold", y=1.01)

    bins_fine = np.linspace(0, CHI2_MAX_DISPLAY, 80)

    for col_idx, (cat, c2) in enumerate(chi2_arrays.items()):
        lbl = CATEGORIES[cat]["label"]
        color = CATEGORIES[cat]["color"]
        ax = axes[0][col_idx]

        # Estimate ndof from median of this category's chi2
        ndof_cat = max(1, int(round(np.median(c2))))
        chi2_x = np.linspace(0.1, CHI2_MAX_DISPLAY, 300)
        # Scale theory curve to match histogram counts
        bin_width = bins_fine[1] - bins_fine[0]
        chi2_theory = stats.chi2.pdf(chi2_x, df=ndof_cat) * len(c2) * bin_width

        ax.hist(
            c2,
            bins=bins_fine,
            alpha=0.70,
            color=color,
            edgecolor="white",
            linewidth=0.4,
            label=f"N={len(c2):,}",
        )
        ax.plot(chi2_x, chi2_theory, "k--", lw=1.5, label=f"$\\chi^2$(ndof={ndof_cat}) scaled")

        med_val = np.median(c2)
        p95_val = np.percentile(c2, 95)
        ax.axvline(med_val, color=color, lw=1.5, ls="-", label=f"median = {med_val:.2f}")
        ax.axvline(p95_val, color=color, lw=1.2, ls="--", label=f"p95 = {p95_val:.2f}")
        ax.axvline(1.0, color="grey", lw=0.8, ls=":", alpha=0.7)

        ax.set_title(lbl, color=color, fontweight="bold", fontsize=10)
        ax.set_xlabel("Dipole fit chi²", fontsize=10)
        if col_idx == 0:
            ax.set_ylabel("N dipoles")
        ax.set_xlim(0, CHI2_MAX_DISPLAY)
        ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    savefig("11d_chi2_per_category")
    plt.show()

## 8. Chi² histograms — per category × per band

3×6 grid (category × band): chi² distribution for dipole-flagged sources in each
photometric band.  PSF quality and template depth vary strongly with band;
this plot reveals band-dependent chi² systematics.

In [ ]:
if chi2_arrays:
    n_cats = len(chi2_arrays)
    n_bands = len(BANDS)
    fig, axes = plt.subplots(
        n_cats,
        n_bands,
        figsize=(3.0 * n_bands, 3.2 * n_cats),
        sharex=True,
        sharey=False,
        squeeze=False,
    )
    fig.suptitle(
        "Dipole fit chi² — per Gaia category × photometric band",
        fontsize=12,
        fontweight="bold",
        y=1.01,
    )

    bins_band = np.linspace(0, CHI2_MAX_DISPLAY, 41)

    for row_idx, cat in enumerate(chi2_arrays):
        lbl = CATEGORIES[cat]["label"]
        color = CATEGORIES[cat]["color"]
        df_d = data_dip[cat]

        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            df_band = df_d[(df_d["r:band"] == band)]
            c2_band = df_band["chi2"].dropna().values if "chi2" in df_band.columns else np.array([])

            if len(c2_band) > 0:
                ax.hist(
                    c2_band,
                    bins=bins_band,
                    alpha=0.70,
                    color=color,
                    edgecolor=bcolor,
                    linewidth=0.8,
                    label=f"N={len(c2_band)}",
                )
                med = np.median(c2_band)
                ax.axvline(med, color="k", lw=1.0, ls="--", label=f"med={med:.1f}")
                ax.legend(fontsize=6, loc="upper right")
            else:
                ax.text(
                    0.5,
                    0.5,
                    "no data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="grey",
                    fontsize=9,
                )

            ax.axvline(1.0, color="grey", lw=0.6, ls=":", alpha=0.6)

            # Column header (band name) on top row
            if row_idx == 0:
                ax.set_title(f"band {band}", color=bcolor, fontweight="bold", fontsize=9)

            # Row label on leftmost column
            if col_idx == 0:
                ax.set_ylabel(f"{lbl}\nN dipoles", color=color, fontsize=8)

            # x-axis label on bottom row
            if row_idx == n_cats - 1:
                ax.set_xlabel("chi²", fontsize=8)

    plt.tight_layout()
    savefig("11d_chi2_per_category_per_band")
    plt.show()

## 9. Cumulative distribution function (CDF) of chi²

CDF comparison across categories, for all bands combined and for the two most
populated bands.  A Kolmogorov–Smirnov test between each pair of categories is
reported to quantify whether the chi² distributions are statistically distinguishable.

In [ ]:
if chi2_arrays:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("CDF of dipole fit chi² — all categories", fontsize=11, fontweight="bold", y=1.01)

    x_max = CHI2_MAX_DISPLAY

    for ax_idx, (ax, band_sel) in enumerate(zip(axes, ["all", "r"])):
        title_str = "All bands" if band_sel == "all" else f"Band {band_sel}"
        ax.set_title(title_str, fontsize=10)

        for cat, c2_all in chi2_arrays.items():
            lbl = CATEGORIES[cat]["label"]
            color = CATEGORIES[cat]["color"]

            if band_sel != "all":
                df_d = data_dip[cat]
                if "chi2" not in df_d.columns:
                    continue
                c2 = df_d[df_d["r:band"] == band_sel]["chi2"].dropna().values
            else:
                c2 = c2_all

            if len(c2) == 0:
                continue

            c2_sorted = np.sort(c2[c2 <= x_max])
            cdf = np.arange(1, len(c2_sorted) + 1) / len(c2_sorted)

            ax.plot(c2_sorted, cdf, color=color, lw=2.0, label=f"{lbl} (N={len(c2):,})", alpha=0.85)

        # Theoretical CDF
        all_c2 = np.concatenate(list(chi2_arrays.values()))
        ndof_ref = max(1, int(round(np.median(all_c2))))
        x_th = np.linspace(0, x_max, 300)
        ax.plot(
            x_th,
            stats.chi2.cdf(x_th, df=ndof_ref),
            "k--",
            lw=1.2,
            alpha=0.7,
            label=f"$\\chi^2$(ndof={ndof_ref}) theory",
        )

        ax.set_xlabel("Dipole fit chi²", fontsize=10)
        ax.set_ylabel("CDF")
        ax.set_xlim(0, x_max)
        ax.set_ylim(0, 1.02)
        ax.axvline(1.0, color="grey", lw=0.7, ls=":", alpha=0.6)
        ax.legend(fontsize=8)

    plt.tight_layout()
    savefig("11d_chi2_CDF_comparison")
    plt.show()

## 10. Kolmogorov–Smirnov pairwise tests

Are the chi² distributions of the three categories statistically different?
The two-sample KS test gives a p-value: small p (< 0.05) means the samples
are drawn from different underlying distributions.

In [ ]:
if len(chi2_arrays) >= 2:
    cats_list = list(chi2_arrays.keys())
    print("Pairwise KS tests on dipole fit chi² (all bands combined):")
    print(f"{'Cat A':40s}  {'Cat B':40s}  {'KS stat':>9}  {'p-value':>12}  {'significant?':>12}")
    print("-" * 120)

    for i in range(len(cats_list)):
        for j in range(i + 1, len(cats_list)):
            catA, catB = cats_list[i], cats_list[j]
            c2A = chi2_arrays[catA]
            c2B = chi2_arrays[catB]
            ks_stat, p_val = stats.ks_2samp(c2A, c2B)
            sig = "YES ***" if p_val < 0.05 else "no"
            lblA = CATEGORIES[catA]["label"]
            lblB = CATEGORIES[catB]["label"]
            print(f"{lblA:40s}  {lblB:40s}  {ks_stat:9.4f}  {p_val:12.4e}  {sig:>12}")

## 11. Chi² vs. scienceFlux (2D histogram)

2D density map of chi² (y) vs. scienceFlux AB magnitude (x) for each category.
This reveals whether high chi² dipoles are preferentially associated with bright
sources (likely PSF-match residuals) or faint sources (noise artefacts).

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy


def flux_nJy_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f / AB_FLUX_ZERO), np.nan)


if chi2_arrays:
    n_cats = len(chi2_arrays)
    fig, axes = plt.subplots(1, n_cats, figsize=(5.5 * n_cats, 5), squeeze=False)
    fig.suptitle(
        "Dipole fit chi²  vs. scienceFlux — 2D density map (dipole sources only)",
        fontsize=11,
        fontweight="bold",
        y=1.01,
    )

    mag_bins = np.arange(14, 28.5, 0.5)
    chi2_bins = np.linspace(0, CHI2_MAX_DISPLAY, 50)

    for col_idx, cat in enumerate(chi2_arrays):
        lbl = CATEGORIES[cat]["label"]
        color = CATEGORIES[cat]["color"]
        ax = axes[0][col_idx]
        df_d = data_dip[cat]

        # Determine science-flux column
        if "r:scienceFlux" in df_d.columns:
            mag_col = "mag_science"
            if "mag_science" not in df_d.columns:
                df_d = df_d.copy()
                df_d["mag_science"] = flux_nJy_to_mag_AB(df_d["r:scienceFlux"].values)
        else:
            mag_col = "mag_psf_abs"
            if "mag_psf_abs" not in df_d.columns:
                df_d = df_d.copy()
                df_d["mag_psf_abs"] = flux_nJy_to_mag_AB(np.abs(df_d["r:psfFlux"].values))

        tmp = df_d[[mag_col, "chi2"]].dropna()
        mag_vals = tmp[mag_col].values
        chi2_vals = tmp["chi2"].values

        # Filter to display range
        mask = (
            (chi2_vals >= chi2_bins[0])
            & (chi2_vals <= chi2_bins[-1])
            & (mag_vals >= mag_bins[0])
            & (mag_vals <= mag_bins[-1])
        )

        if mask.sum() > 10:
            h, xedges, yedges = np.histogram2d(mag_vals[mask], chi2_vals[mask], bins=[mag_bins, chi2_bins])
            im = ax.pcolormesh(xedges, yedges, h.T, cmap="viridis", shading="auto")
            plt.colorbar(im, ax=ax, label="N dipoles", fraction=0.04, pad=0.02)

        ax.axhline(1.0, color="white", lw=0.8, ls=":", alpha=0.8, label="chi2=1")
        ax.set_title(lbl, color=color, fontweight="bold", fontsize=10)
        ax.set_xlabel("scienceFlux (AB mag)" if "science" in mag_col else "|psfFlux| (AB mag)")
        if col_idx == 0:
            ax.set_ylabel("Dipole fit chi²")
        ax.set_ylim(0, CHI2_MAX_DISPLAY)
        ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    savefig("11d_chi2_vs_scienceFlux_2D")
    plt.show()

## 12. Reduced chi² interpretation

If the number of degrees of freedom (`ndof`) of the dipole fit is known or
can be inferred from the data, we compute the reduced chi² = chi² / ndof.

For `ip_diffim` dipole fits in the Rubin stack, the fit has 5 free parameters
(centroid x, centroid y, orientation, positive flux, negative flux) subtracted
from the number of pixels in the fit stamp.  A typical 21×21 pixel stamp gives
`ndof ≈ 441 − 5 = 436`.  In the Fink parquet, however, the chi² is often
already normalised per pixel (i.e. it is already a reduced chi²).  We cannot
determine this from the data alone.  We therefore present both the raw chi²
and, if `ndof_est` > 1, the re-scaled quantity chi² / ndof_est.

In [ ]:
# ── Optional: set ndof manually if you know it ────────────────────────────────
# Example: NDOF = 436   # 21x21 stamp minus 5 fit parameters
# For now we estimate from the median of the pooled chi2 distribution.

if chi2_arrays:
    all_c2 = np.concatenate(list(chi2_arrays.values()))
    NDOF_EST = max(1, int(round(np.median(all_c2))))
    print(f"Estimated ndof from pooled median: {NDOF_EST}")
    print("(If chi2 is already a per-pixel reduced chi2, NDOF_EST should be ~1.)")

    rows_red = []
    for cat, c2 in chi2_arrays.items():
        lbl = CATEGORIES[cat]["label"]
        c2r = c2 / NDOF_EST
        rows_red.append(
            {
                "category": lbl,
                "ndof_est": NDOF_EST,
                "mean_chi2_red": round(float(np.mean(c2r)), 4),
                "median_chi2_red": round(float(np.median(c2r)), 4),
                "frac_chi2_red_gt2": round(float((c2r > 2).mean()), 4),
                "frac_chi2_red_gt5": round(float((c2r > 5).mean()), 4),
            }
        )

    print("\nReduced chi² summary (chi²_raw / ndof_est):")
    display(pd.DataFrame(rows_red))

## 13. Chi² heatmap — per category × band (median chi²)

Heatmap showing the **median chi²** of dipole-flagged sources in each (category, band)
cell.  Bright colours indicate bands/categories where dipole fits are systematically
worse (high chi²).

In [ ]:
if chi2_arrays:
    hmap_rows = []
    for cat in chi2_arrays:
        df_d = data_dip[cat]
        lbl = CATEGORIES[cat]["label"]
        row = {"category": lbl}
        for band in BANDS:
            if "chi2" not in df_d.columns:
                row[band] = np.nan
                continue
            c2b = df_d[df_d["r:band"] == band]["chi2"].dropna().values
            row[band] = float(np.median(c2b)) if len(c2b) > 0 else np.nan
        hmap_rows.append(row)

    df_hmap = pd.DataFrame(hmap_rows).set_index("category")

    fig, ax = plt.subplots(figsize=(9, 2.0 + 0.9 * len(chi2_arrays)))
    im = ax.imshow(
        df_hmap.values,
        aspect="auto",
        cmap="YlOrRd",
        vmin=0,
        vmax=np.nanpercentile(df_hmap.values, 95),
    )

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels(BANDS, fontsize=11)
    ax.set_yticks(range(len(df_hmap)))
    ax.set_yticklabels(df_hmap.index, fontsize=9)
    ax.set_title("Median dipole fit chi² per category × band", fontweight="bold", pad=10)

    for i in range(len(df_hmap)):
        for j, band in enumerate(BANDS):
            val = df_hmap.values[i, j]
            if np.isfinite(val):
                threshold = np.nanpercentile(df_hmap.values, 70)
                tc = "white" if val > threshold else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10, color=tc)
            else:
                ax.text(j, i, "—", ha="center", va="center", fontsize=10, color="grey")

    cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Median chi²")

    plt.tight_layout()
    savefig("11d_chi2_heatmap_median")
    plt.show()

    print("\nNumerical heatmap:")
    display(df_hmap.round(3))

## 14. Conclusions

Key questions this notebook answers:

1. **Are the dipole chi² distributions different across Gaia categories?**  
   Sections 6–7 (overlaid histograms) and Section 10 (KS tests) address this directly.
   If stable stars show higher chi² than variable stars, it suggests their dipoles
   arise from poor PSF-match residuals rather than genuine source variability.

2. **Is the chi² distribution consistent with the theoretical chi² distribution?**  
   The theoretical curve overlay (Sections 6–7) and the CDF comparison
   (Section 9) test this.  A heavy tail (chi² >> ndof) indicates poor fits.

3. **Are high-chi² dipoles concentrated at bright or faint magnitudes?**  
   The 2D histogram chi² vs. scienceFlux (Section 11) answers this.  Bright sources
   generating high chi² are prime candidates for Butler stamp inspection.

4. **Which bands show the worst dipole fits?**  
   The per-band heatmap (Section 13) and the category × band grid (Section 8) reveal
   band-dependent chi² systematics, potentially linked to PSF model quality per band.

> **Recommended next step:** cross-match the objects with highest chi² (tails of the
> distributions) with the top-dipole-dominated objects from notebook 11 (Section 12)
> and inspect their Butler difference-image stamps for visual confirmation of the
> dipole or non-dipole morphology.

In [ ]:
print("Notebook 11d — dipole chi² analysis — complete.")